# Open Ended Practical: Smart Parking Slot Detection System

**Author:** Yash Kapse  
**Subject:** Machine Vision / Computer Vision Practical  

---  
## 📌 Project Overview & Problem Statement
Urban parking management requires automated systems to monitor parking space availability in real time. 

This practical implements a **Smart Parking Slot Detection System** using Computer Vision techniques in OpenCV. The system:
1. Accepts an aerial video feed or static image of a parking lot.
2. Defines bounding box Regions of Interest (ROIs) for individual parking bays `[x, y, w, h]`.
3. Preprocesses each ROI using **Grayscale Conversion**, **Gaussian Denoising**, **Adaptive Binarization**, and **Canny Edge Detection**.
4. Counts non-zero thresholded pixels in each ROI to classify slots as **Occupied 🔴** (pixel density > threshold) or **Vacant 🟢** (pixel density ≤ threshold).
5. Computes real-time parking metrics (**Total Slots**, **Occupied Slots**, **Available Slots**, **Occupancy Rate %**).

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add core module directory to sys.path
sys.path.append(os.path.abspath('.'))
from core.parking_detector import ParkingDetector
from core.slot_picker import SlotManager
from samples.generate_parking_lot import generate_parking_lot_image

%matplotlib inline
print("✓ Machine Vision modules loaded successfully!")

--- 
## 1. Input Parking Lot Dataset & ROI Layout Generation
We generate a synthetic aerial parking lot image containing 12 parking slots with 6 occupied spaces (cars parked) and 6 vacant spaces.

In [ ]:
# Generate sample parking lot image with occupied slots [0, 2, 3, 5, 7, 10]
raw_img, slots, metadata = generate_parking_lot_image(rows=2, cols=6, occupied_indices=[0, 2, 3, 5, 7, 10])

print(f"Parking Lot Dimensions: {raw_img.shape[1]}x{raw_img.shape[0]} px")
print(f"Ground Truth Metadata: {metadata}")

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))
plt.title("Input Parking Lot Image", fontsize=14)
plt.axis("off")
plt.show()

--- 
## 2. Image Preprocessing Pipeline (Grayscale, Blur, Adaptive Thresholding)
To detect vehicles reliably regardless of car color or shadows, we convert the image to grayscale, apply Gaussian blur to suppress noise, and use **Adaptive Gaussian Thresholding** to convert vehicle edges and textures into non-zero pixel clusters.

In [ ]:
detector = ParkingDetector(pixel_threshold=800)
gray_img, binary_img = detector.preprocess_frame(raw_img, blur_kernel=3, block_size=25, c_val=16)

# Compute Canny edges for visual comparison
canny_edges = cv2.Canny(gray_img, 50, 150)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(gray_img, cmap='gray')
axes[0].set_title("1. Grayscale Image", fontsize=12)
axes[0].axis("off")

axes[1].imshow(binary_img, cmap='gray')
axes[1].set_title("2. Adaptive Threshold (Binarized)", fontsize=12)
axes[1].axis("off")

axes[2].imshow(canny_edges, cmap='gray')
axes[2].set_title("3. Canny Edge Detection", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

--- 
## 3. Slot ROI Classification (Pixel Density Analysis)
For each slot ROI `[x, y, w, h]`, `cv2.countNonZero(roi)` computes the pixel density. 
- If `non_zero_count > 800` -> **Occupied 🔴** (Draw Red Rectangle)
- If `non_zero_count <= 800` -> **Vacant 🟢** (Draw Green Rectangle)

In [ ]:
results = detector.process_slots(raw_img, slots)

print("=== PARKING OCCUPANCY SUMMARY ===")
print(f"Total Parking Bays : {results['total_slots']}")
print(f"Occupied Slots (🔴) : {results['occupied_slots']}")
print(f"Available Slots(🟢) : {results['available_slots']}")
print(f"Occupancy Rate (%) : {results['occupancy_rate']}%")

# Visualize Final Classified Bounding Box Overlay
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(results['overlay_image'], cv2.COLOR_BGR2RGB))
plt.title("Classified Parking Slots (Green: Vacant 🟢, Red: Occupied 🔴)", fontsize=14)
plt.axis("off")
plt.show()

--- 
## 4. Slot Breakdown Table & Pixel Density Inspection

In [ ]:
print(f"{'Slot ID':<10} | {'Status':<12} | {'Non-Zero Pixel Count':<22} | {'Threshold':<10}")
print("-" * 65)
for s in results['slots']:
    icon = "🔴" if s['occupied'] else "🟢"
    print(f"Slot {s['id']:<5} | {icon} {s['status']:<9} | {s['non_zero_pixels']:<22} | {detector.pixel_threshold:<10}")

--- 
## 5. Machine Vision Practical & Viva Q&A Guide

### Q1: Why do we use Adaptive Thresholding instead of Global Thresholding for parking slot detection?
**Answer:** Global thresholding applies a fixed pixel intensity cut-off across the entire image. Parking lots experience non-uniform outdoor lighting, passing shadows, and sun glare. Adaptive thresholding calculates localized thresholds for small pixel neighborhoods, making edge/feature extraction invariant to global illumination changes.

### Q2: How does pixel density counting differentiate a parked car from an empty parking bay?
**Answer:** An empty parking bay consists of uniform, smooth asphalt texture with very few edges, producing low non-zero pixel counts (e.g. < 300). A parked vehicle introduces complex geometry (windshields, body contours, mirrors, tires), producing high edge density and high non-zero pixel counts (e.g. > 1200).

### Q3: How can this system be extended to handle video streams?
**Answer:** The `process_slots()` function can be executed inside a video frame loop (`while cap.isOpened(): ret, frame = cap.read()`). To reduce frame-by-frame flickering, a temporal moving average or consecutive frame confirmation counter (e.g., slot status updated only if consistent for 5 consecutive frames) can be added.